# 🤖 RAG Chatbot — Recent NLP/LLM Papers + Gemma 3 4b

This notebook launches a Gradio chat UI that:
1. Embeds your question locally with `sentence-transformers`
2. Retrieves the top-k most relevant chunks from ChromaDB
3. Injects those chunks as context into Gemma 3 4b via Ollama
4. Streams the grounded answer back to the UI with paper citations

> **Prerequisites:**
> - Run `ingest.ipynb` first to populate `./chroma_db/`
> - Have Ollama running: `ollama serve` + `ollama pull gemma3:4b`

In [ ]:
%pip install -q chromadb sentence-transformers gradio requests

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
CHROMA_DIR      = "./chroma_db"
COLLECTION_NAME = "arxiv_nlp"
EMBED_MODEL     = "all-MiniLM-L6-v2"
TOP_K           = 5           # number of chunks to retrieve per query

OLLAMA_URL  = "http://localhost:11434/api/chat"
MODEL_NAME  = "gemma3:4b"

SYSTEM_PROMPT = (
    "You are a great general model but as you've not been trained on recent research I'm helping you by extracting that data and giving it to you.\n"
    "Think of yourself as a human using a book to answer the query.While the book just provides extracted information,you understand, reason and answer.\n"
    "If the context does not contain enough information, say so clearly.\n"
    "Be concise and precise. When citing a paper, use its title or arXiv ID.\n"
    "The answer isn't all about the paper and what you get from the content i give. You are the star of the show, the response should be a clean logical answer that you give using the content i provide. What I'm saying is if the content I'm giving you is a book, you don't just narrate it, you understand it.\n"
    "Explain the answer in points if needed or a paragraph. Don't just give the answer and stop, explain like a teacher clarifying the answer to a student who has doubts. Don't explicitly say explanation for the student."
)

In [ ]:
# ── Imports & setup ───────────────────────────────────────────────────────────
import json
import warnings
import chromadb
import gradio as gr
import requests
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

print(f"Loading embedder '{EMBED_MODEL}'...")
embedder = SentenceTransformer(EMBED_MODEL)

print(f"Connecting to ChromaDB at '{CHROMA_DIR}'...")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' — {collection.count():,} chunks ready ✓")

In [ ]:
# ── RAG helpers ───────────────────────────────────────────────────────────────
def retrieve(query: str, k: int = TOP_K) -> tuple[list[str], list[dict]]:
    """Return (chunks, metadatas) for the top-k results."""
    q_emb = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    return results["documents"][0], results["metadatas"][0]


def build_context_block(chunks: list[str], metadatas: list[dict]) -> str:
    """Format retrieved chunks with metadata for the LLM prompt."""
    parts = []
    for i, (chunk, meta) in enumerate(zip(chunks, metadatas)):
        title   = meta.get("title", "Unknown")
        pub_date = meta.get("pub_date", "")
        arxiv_id = meta.get("paper_id", "")
        parts.append(
            f"[Passage {i+1}] [{arxiv_id} | {pub_date}] {title}\n{chunk}"
        )
    return "\n\n".join(parts)


def build_sources_md(chunks: list[str], metadatas: list[dict]) -> str:
    """Markdown block shown above the answer listing source papers."""
    lines = [f"**📄 {len(chunks)} sources retrieved:**\n"]
    seen_papers = set()
    for meta, chunk in zip(metadatas, chunks):
        pid = meta.get("paper_id", "")
        if pid in seen_papers:
            continue
        seen_papers.add(pid)
        title    = meta.get("title", "Unknown title")
        pub_date = meta.get("pub_date", "")
        url      = meta.get("url", "")
        preview  = chunk[:120].replace("\n", " ").strip()
        link     = f"[{title}]({url})" if url else title
        lines.append(f"- **{link}** ({pub_date})  \n  *{preview}...*")
    return "\n".join(lines) + "\n\n---\n\n"


def history_to_messages(history: list[dict]) -> list[dict]:
    """Convert Gradio messages-format history to Ollama messages list."""
    messages = []
    for turn in history:
        role    = turn.get("role", "")
        content = turn.get("content", "")
        if isinstance(content, list):   # multimodal safety
            content = " ".join(
                b.get("text", "") for b in content if isinstance(b, dict)
            )
        content = content.strip()
        if role in ("user", "assistant") and content:
            messages.append({"role": role, "content": content})
    return messages


print("RAG helpers ready ✓")

In [ ]:
# ── Ollama streaming call ─────────────────────────────────────────────────────
def chat_with_rag(history: list[dict], show_sources: bool = True):
    """Generator that yields incrementally updated assistant response strings."""
    # Convert history (excluding the empty placeholder assistant turn)
    prior_history = history[:-1]   # last entry is the empty assistant placeholder
    messages = history_to_messages(prior_history)

    if not messages:
        yield "Please type a question first."
        return

    # Latest user message is used for retrieval
    latest_user_msg = next(
        (m["content"] for m in reversed(messages) if m["role"] == "user"), ""
    )
    if not latest_user_msg:
        yield "Could not find a user message to process."
        return

    # Retrieve from ChromaDB
    chunks, metadatas = retrieve(latest_user_msg)
    context_block = build_context_block(chunks, metadatas)

    grounded_system = (
        f"{SYSTEM_PROMPT}\n\n"
        f"=== Retrieved Context (papers from 2025) ===\n"
        f"{context_block}\n"
        f"============================================"
    )

    payload = {
        "model"   : MODEL_NAME,
        "messages": [{"role": "system", "content": grounded_system}, *messages],
        "stream"  : True,
    }

    preamble = build_sources_md(chunks, metadatas) if show_sources else ""

    try:
        resp = requests.post(OLLAMA_URL, json=payload, stream=True, timeout=120)
        resp.raise_for_status()
        full_response = preamble
        for line in resp.iter_lines():
            if line:
                chunk_data = json.loads(line.decode("utf-8"))
                token = chunk_data.get("message", {}).get("content", "")
                full_response += token
                yield full_response

    except requests.exceptions.ConnectionError:
        yield (
            "**❌ Cannot connect to Ollama.**\n\n"
            "Make sure it's running:\n"
            "```\nollama serve\nollama pull gemma3:4b\n```"
        )
    except requests.exceptions.HTTPError as e:
        yield f"**❌ Ollama HTTP error:** {e}\n\nIs `gemma3:4b` pulled? Run `ollama pull gemma3:4b`"
    except Exception as e:
        yield f"**❌ Unexpected error:** {str(e)}"


print("chat_with_rag ready ✓")

In [ ]:
# ── Gradio UI ─────────────────────────────────────────────────────────────────
THEME = gr.themes.Default(
    primary_hue="violet",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter"),
)

CSS = """
.gradio-container {
    max-width: 100% !important;
    margin: 0 auto;
    padding-top: 0 !important;
}
.gradio-container .main { min-height: 100vh; }
#app-shell {
    min-height: 100vh;
    display: flex;
    flex-direction: column;
    gap: 0.75rem;
    padding: 0.75rem 1rem 1rem;
}
#header-row { display: flex; justify-content: flex-end; align-items: center; gap: 0.6rem; }
#header-row h1 { font-size: 1.1rem; font-weight: 700; margin: 0;
    background: linear-gradient(135deg, #7c3aed, #a78bfa);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
.rag-badge { font-size: 0.7rem; font-weight: 700; padding: 2px 8px;
    background: #ede9fe; color: #6d28d9; border-radius: 99px; }
#main-grid {
    flex: 1 1 auto;
    align-items: stretch;
    gap: 1rem;
}
#chat-pane {
    flex: 1 1 auto;
    min-height: calc(100vh - 250px);
    display: flex;
    flex-direction: column;
    gap: 0.75rem;
}
#chatbox {
    border: none !important;
    background: transparent !important;
    box-shadow: none !important;
    flex: 1 1 auto;
    min-height: calc(100vh - 320px);
}
#chatbox .wrap { min-height: inherit; }
#chatbox .overflow-y-auto { min-height: inherit; }
#input-row { align-items: flex-end; gap: 8px; }
#msg-box textarea { border-radius: 12px !important; font-size: 0.97rem; min-height: 54px; }
#send-btn { min-width: 88px; border-radius: 12px !important; }
#bottom-grid {
    gap: 1rem;
    align-items: stretch;
}
#bottom-card {
    border: 1px solid rgba(148, 163, 184, 0.18);
    border-radius: 18px;
    padding: 0.9rem;
    background: rgba(15, 23, 42, 0.28);
    backdrop-filter: blur(10px);
    min-height: 180px;
}
#panel-title {
    margin: 0 0 0.55rem 0;
    font-size: 0.92rem;
    font-weight: 700;
    color: #e2e8f0;
}
#footer { text-align: center; color: #475569; font-size: 0.78rem; padding: 0.4rem 0 0; }
@media (max-width: 980px) {
    #chat-pane { min-height: auto; }
    #chatbox { min-height: 520px; }
    #bottom-grid { flex-direction: column; }
    #bottom-card { min-height: auto; }
}
"""

EXAMPLE_QUESTIONS = [
    "What techniques improve in-context learning in LLMs?",
    "How does chain-of-thought prompting work?",
    "What are recent approaches to reducing hallucination?",
    "Explain RLHF and how it aligns language models.",
    "What is the difference between sparse and dense retrieval?",
    "What new LLM architectures were proposed in early 2025?",
]


def build_ui():
    with gr.Blocks(theme=THEME, css=CSS, fill_height=True) as demo:

        with gr.Column(elem_id="app-shell"):

            with gr.Row(elem_id="header-row"):
                gr.HTML('<span class="rag-badge">RAG · 2025 Papers</span>'
                        '<h1>Gemma 3 4b + arXiv NLP</h1>')

            with gr.Column(elem_id="chat-pane"):
                # Chatbot now uses the message dict format directly in Gradio 6.x.
                chatbot = gr.Chatbot(
                    elem_id="chatbox",
                    height=720,
                    show_label=False,
                    render_markdown=True,
                )

                with gr.Row(elem_id="input-row"):
                    msg = gr.Textbox(
                        placeholder="Ask anything about 2025 NLP/LLM research…",
                        show_label=False,
                        elem_id="msg-box",
                        scale=9,
                        autofocus=True,
                    )
                    send = gr.Button("Send", variant="primary", elem_id="send-btn", scale=1)

            with gr.Row(elem_id="bottom-grid"):
                with gr.Column(elem_id="bottom-card"):
                    gr.Markdown("""<div id="panel-title">Retrieval Controls</div>""")
                    show_sources = gr.Checkbox(
                        value=True,
                        label="Show retrieved source passages",
                    )
                    clear = gr.Button("Clear conversation", variant="secondary", size="sm")

                with gr.Column(elem_id="bottom-card"):
                    gr.Markdown("""<div id="panel-title">Try a research question</div>""")
                    gr.Examples(
                        examples=EXAMPLE_QUESTIONS,
                        inputs=msg,
                        label=None,
                    )

            gr.HTML(
                '<div id="footer">'
                "Grounded on arXiv cs.CL / cs.AI papers (2025) · Running 100% locally"
                "</div>"
            )

        # ── Event handlers ────────────────────────────────────────────────────
        def user_turn(user_msg: str, history: list):
            """Append user message and empty assistant placeholder."""
            history = history or []
            if not user_msg.strip():
                return "", history
            history = history + [
                {"role": "user",      "content": user_msg},
                {"role": "assistant", "content": ""},
            ]
            return "", history

        def bot_turn(history: list, sources_on: bool):
            """Stream the assistant response into the last history entry."""
            if not history:
                return history
            for partial in chat_with_rag(history, show_sources=sources_on):
                history[-1]["content"] = partial
                yield history

        submit_event = msg.submit(
            user_turn, [msg, chatbot], [msg, chatbot], queue=False
        ).then(bot_turn, [chatbot, show_sources], chatbot)

        send.click(
            user_turn, [msg, chatbot], [msg, chatbot], queue=False
        ).then(bot_turn, [chatbot, show_sources], chatbot)

        clear.click(lambda: [], None, chatbot, queue=False)

    return demo


demo = build_ui()
demo.queue()

# Try ports 7861–7870 in case one is busy
for port in range(7861, 7871):
    try:
        demo.launch(
            server_name="127.0.0.1",
            server_port=port,
            share=False,
            show_error=True,
        )
        break
    except OSError:
        continue